# Tracking and metrics (milestone 3)

This notebook runs a scenario (an hour of truth, one rotating 2D radar and
three Stone Soup trackers) and compares the trackers on the same
detections:

| Tracker | Filter | Motion model |
| --- | --- | --- |
| `ekf-cv-gnn` | extended Kalman (the baseline) | constant velocity |
| `ukf-cv-gnn` | unscented Kalman | constant velocity |
| `pf-mm-gnn` | multiple-model particle filter | constant velocity or coordinated turn |

All three associate with global nearest neighbour, start tracks at an
assumed height (a 2D radar measures no height) and confirm them after three
detections. The run takes about a minute; `tracking-sim run` does the same
from the command line.

With `TRACKING_SIM_RECORDING` set, the scenario's sensors and trackers run on
your capture instead of its truth file.

| Variable | Meaning |
| --- | --- |
| `TRACKING_SIM_SCENARIO` | Scenario YAML (default `scenarios/kbjc-synthetic.yaml`) |
| `TRACKING_SIM_SEED` | Override the scenario's seed |
| `TRACKING_SIM_RECORDING` | A capture to use instead: a recording (`.parquet`/`.jsonl`) or SBS text (`.sbs`, `.sbs.gz`) |
| `TRACKING_SIM_WINDOW` | `START,END` in ISO 8601 UTC, to use part of the capture (default: all of it) |
| `TRACKING_SIM_SBS_TZ` | Time zone of SBS timestamps (Dump1090 writes host local time; default UTC) |

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tracking_sim.cli import parse_time
from tracking_sim.eval.scoring import segment_of
from tracking_sim.sim.scenario import load_scenario, run_scenario

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SCENARIO = os.environ.get("TRACKING_SIM_SCENARIO") or str(ROOT / "scenarios/kbjc-synthetic.yaml")
RECORDING = os.environ.get("TRACKING_SIM_RECORDING")
WINDOW = os.environ.get("TRACKING_SIM_WINDOW")
SBS_TZ = os.environ.get("TRACKING_SIM_SBS_TZ")

# Colours: the reference data-viz palette's first three categorical slots, plus neutrals.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
BLUE, ORANGE = SERIES[0], SERIES[1]
INK, GREY, LIGHT = "#52514e", "#8a8984", "#c9c8c2"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

scenario = load_scenario(SCENARIO)
if os.environ.get("TRACKING_SIM_SEED"):
    scenario = scenario.with_seed(int(os.environ["TRACKING_SIM_SEED"]))
if RECORDING:
    window = tuple(parse_time(t) for t in WINDOW.split(",")) if WINDOW else None
    scenario = scenario.with_truth(RECORDING, window=window, tz=SBS_TZ)
print("scenario:", SCENARIO)
print("truth:   ", scenario.truth_path)
print("trackers:", ", ".join(t.id for t in scenario.trackers))

## 1. Run it

Truth is built once, the radar is simulated once, and every tracker gets
the same detections. Scoring uses only the truth the radar could see:
an aircraft counts at a scan only if it passed the range, elevation and
horizon gates during that scan.

In [ ]:
result = run_scenario(scenario)
print(result.summary())
colour = {run.tracker_id: SERIES[i % len(SERIES)] for i, run in enumerate(result.runs)}

**Reading the table.** Lower is better for OSPA, GOSPA and its parts
(localisation error, missed aircraft, false tracks), ambiguity (tracks per
aircraft held), spuriousness (share of tracks on no aircraft), position
accuracy, latency (seconds from an aircraft becoming visible to being
tracked) and *never* (aircraft never tracked). Higher is better for
completeness (share of visible aircraft held by a track) and *longest*
(share of each aircraft's visible time on its single longest track).
OSPA is a per-aircraft average in metres, capped at the 1 km cut-off;
GOSPA is a total across aircraft, so it grows with traffic.

## 2. Tracks over truth

Confirmed tracks from each tracker over the truth (grey). A track whose
updates came mostly from false alarms is drawn in orange: a false track.

In [ ]:
def track_kinds(run):
    table = run.tracks_table().to_pandas()
    table = table[table["confirmed"]]
    updates = table[table["updated"]]
    clutter_share = (updates["detection_kind"] == "clutter").groupby(updates["track_id"]).mean()
    false_ids = set(clutter_share[clutter_share > 0.5].index)
    return table, false_ids


fig, axes = plt.subplots(
    1, len(result.runs), figsize=(5.2 * len(result.runs), 5.4), sharex=True, sharey=True
)
site = result.simulation.radars[scenario.sensors[0].id].site.position / 1e3
for ax, run in zip(np.atleast_1d(axes), result.runs):
    for seg in result.truth:
        t = np.arange(seg.t[0], seg.t[-1], 5.0)
        pos, _, ok = seg.evaluate(t)
        ax.plot(pos[ok, 0] / 1e3, pos[ok, 1] / 1e3, color=LIGHT, lw=2.5)
    table, false_ids = track_kinds(run)
    for track_id, rows in table.groupby("track_id"):
        is_false = track_id in false_ids
        ax.plot(
            rows["east_m"] / 1e3,
            rows["north_m"] / 1e3,
            lw=0.9,
            color=ORANGE if is_false else BLUE,
            zorder=3 if is_false else 2,
        )
    ax.plot(*site[:2], "^", color=INK, ms=6)
    ax.add_patch(
        plt.Circle(
            site[:2],
            scenario.sensors[0].coverage.range_max_m / 1e3,
            fill=False,
            color=GREY,
            lw=0.6,
            ls=":",
        )
    )
    n_tracks = table["track_id"].nunique()
    ax.set(title=f"{run.tracker_id}: {n_tracks} tracks, {len(false_ids)} false", xlabel="east (km)")
    ax.set_aspect("equal")
np.atleast_1d(axes)[0].set_ylabel("north (km)")
handles = [
    plt.Line2D([], [], color=LIGHT, lw=2.5, label="truth"),
    plt.Line2D([], [], color=BLUE, lw=1, label="track"),
    plt.Line2D([], [], color=ORANGE, lw=1, label="false track"),
]
np.atleast_1d(axes)[0].legend(handles=handles, loc="lower left", fontsize=8, frameon=False)
fig.tight_layout()

## 3. Metrics over time

OSPA every scan (left; one minute rolling mean) and the average GOSPA split
into its parts (right): localisation error on tracked aircraft, the cost of
missed aircraft, and the cost of false tracks.

In [ ]:
fig, (ax, ax_g) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1.6, 1]})
t0 = result.grid.times[0]
for score in result.scores:
    minutes = np.array([(t - t0).total_seconds() / 60 for t in score.times])
    rolling = np.convolve(score.ospa, np.ones(12) / 12, mode="valid")
    ax.plot(
        minutes[11:],
        rolling,
        color=colour[score.tracker_id],
        lw=1.6,
        label=f"{score.tracker_id} (mean {score.mean_ospa_m:.0f} m)",
    )
ax.set(xlabel="minutes", ylabel="OSPA (m)", ylim=(0, 1000), title="OSPA, 1 km cut-off")
ax.legend(fontsize=8, frameon=False)

parts = ["localisation", "missed", "false"]
width = 0.8 / len(result.scores)
for i, score in enumerate(result.scores):
    values = [score.mean_gospa(p) for p in parts]
    x = np.arange(len(parts)) + (i - (len(result.scores) - 1) / 2) * width
    ax_g.bar(x, values, width * 0.92, color=colour[score.tracker_id], label=score.tracker_id)
ax_g.set_xticks(range(len(parts)), parts)
ax_g.set(ylabel="mean GOSPA part", title="Where the GOSPA comes from")
ax_g.legend(fontsize=8, frameon=False)
fig.tight_layout()

## 4. One aircraft through a turn

The aircraft that turns the most while the radar can see it: every
confirmed track each tracker held on it, and their error over time. A
constant-velocity filter lags in a turn and can lose the aircraft; the
multiple-model filter can switch particles to a coordinated turn. Even in
straight flight the error has a floor of a few hundred metres here: at
this range the assumed 5,000 m height is wrong enough to bias ground range,
the price of tracking with a 2D radar.

Each spike in the error is a track coasting on after losing the aircraft
in a turn, until it is deleted and a new track starts; the number of tracks
each tracker needed for this one aircraft (in the legend) says how often
that happened.

In [ ]:
def turning(seg):
    t = np.arange(seg.t[0], seg.t[-1], 5.0)
    _, vel, ok = seg.evaluate(t)
    heading = np.unwrap(np.arctan2(vel[ok, 0], vel[ok, 1]))
    return np.max(np.abs(np.diff(heading))) / 5.0 if len(heading) > 2 else 0.0


seen = {segment_of(p.id) for p in result.grid.truths}
seg = max((s for s in result.truth if s.segment_id in seen), key=turning)
paths = [p for p in result.grid.truths if segment_of(p.id) == seg.segment_id]
times = sorted(s.timestamp for p in paths for s in p.states)
print(f"{seg.segment_id} ({seg.callsign}): max turn rate {np.degrees(turning(seg)):.1f} deg/s")

fig, (ax, ax_e) = plt.subplots(1, 2, figsize=(14, 4.8))
t = np.array([x.timestamp() for x in times])
pos, _, _ = seg.evaluate(t)
ax.plot(pos[:, 0] / 1e3, pos[:, 1] / 1e3, color=LIGHT, lw=4, label="truth")
for run in result.runs:
    table = run.tracks_table().to_pandas()
    table = table[table["confirmed"]]
    updates = table[table["updated"]]
    share = (updates["segment_id"] == seg.segment_id).groupby(updates["track_id"]).mean()
    mine = share[share > 0.5].index
    for k, (_track_id, rows) in enumerate(table[table["track_id"].isin(mine)].groupby("track_id")):
        label = f"{run.tracker_id} ({len(mine)} tracks)" if k == 0 else None
        ax.plot(
            rows["east_m"] / 1e3,
            rows["north_m"] / 1e3,
            color=colour[run.tracker_id],
            lw=1.2,
            label=label,
        )
        ts = np.array([x.timestamp() for x in rows["timestamp"]])
        truth_pos, _, ok = seg.evaluate(ts)
        err = np.hypot(
            rows["east_m"].to_numpy() - truth_pos[:, 0],
            rows["north_m"].to_numpy() - truth_pos[:, 1],
        )
        ax_e.plot((ts[ok] - t[0]) / 60, err[ok], color=colour[run.tracker_id], lw=1.2, label=label)
ax.set(
    xlabel="east (km)",
    ylabel="north (km)",
    title=f"{seg.callsign or seg.segment_id}: tracks through the turn",
)
ax.set_aspect("equal")
ax.legend(fontsize=8, frameon=False)
ax_e.set(xlabel="minutes", ylabel="horizontal error (m)", title="Track error on this aircraft")
ax_e.legend(fontsize=8, frameon=False)
fig.tight_layout()

## 5. The comparison

The same table `tracking-sim run` prints. Change a tracker spec in
`scenarios/trackers/` (motion noise, gate, particle count, initiation) or
the radar spec, and re-run to see the effect on identical traffic.

In [ ]:
print(result.table())
best = min(result.scores, key=lambda s: s.mean_ospa_m)
print(f"\nlowest mean OSPA: {best.tracker_id} ({best.mean_ospa_m:.0f} m)")